In [ ]:
from sqlalchemy import create_engine
from sqlalchemy import URL
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
server = 'sharktowels.duckdns.org'
database = "Spending"
user = "SA"

with open("password.txt", "r") as file:
    password = file.read().strip()

conn_string = URL.create(
    "mssql+pyodbc",
    username = user,
    password = password,
    host = server,
    port = 1433,
    database = database,
    query = {
            "driver": "ODBC Driver 18 for SQL Server",
            "TrustServerCertificate": "yes",
            }
)

conn = create_engine(conn_string)


In [33]:
df = pd.read_sql_query(f'''
                        SELECT PasswordHash FROM Credentials
                        ''', conn)

print(df)

temptoken = str(df.iloc[0,0])
print(temptoken)
token = temptoken

                                        PasswordHash
0  81b637d8fcd2c6da6359e6963113a1170de795e4b725b8...
81b637d8fcd2c6da6359e6963113a1170de795e4b725b84d1e0b4cfd9ec58ce9


c:\Users\jduen\AppData\Local\Programs\Python\Python311\Lib\site-packages\pandas\io\sql.py:1636: SAWarning:

Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.



In [34]:
df = pd.read_sql_query(f'''
                        SELECT Users.UserID
                        FROM Users
                        JOIN Credentials ON Users.UserID = Credentials.UserID
                        WHERE Credentials.PasswordHash = '{token}'
                        ''', conn)

userid = str(df.iloc[0,0])

In [31]:
area_df = pd.read_sql_query(f'''
                            SELECT Purchases.TimeDate, Purchases.Category, Purchases.Subcategory, Payments.Amount 
                            FROM Purchases
                            JOIN Payments ON Purchases.PaymentID = Payments.PaymentID
                            WHERE UserID = {userid}
                            ORDER BY Purchases.TimeDate
                            ''', conn)

area_df["runningAmount"] = area_df.sort_values("TimeDate") \
                         .groupby("Category")["Amount"] \
                         .cumsum()

pivoted_area_df = area_df.pivot_table(
    index="TimeDate",
    columns="Category",
    values="runningAmount",
    aggfunc="last"
)

pivoted_area_df = pivoted_area_df.ffill()

pivoted_area_df.head()

fig = go.Figure()

for category in pivoted_area_df.columns:
    fig.add_trace(
        go.Scatter(
            x=pivoted_area_df.index,
            y=pivoted_area_df[category],
            mode="lines",
            stackgroup="one",
            name=category
        )
    )

fig.update_layout(
    title="Cumulative Spending By Category",
    xaxis_title="Time",
    yaxis_title="Running Amount",
    hovermode="x unified",
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1,
                     label="1m",
                     step="month",
                     stepmode="backward"),
                dict(count=6,
                     label="6m",
                     step="month",
                     stepmode="backward"),
                dict(count=1,
                     label="YTD",
                     step="year",
                     stepmode="todate"),
                dict(count=1,
                     label="1y",
                     step="year",
                     stepmode="backward"),
                dict(step="all")
            ])
        ),
    )
)

fig.show()

C:\Users\jduen\AppData\Local\Temp\ipykernel_7384\959971142.py:1: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



In [ ]:
area_df = pd.read_sql_query(f'''
                            SELECT Purchases.TimeDate, Purchases.Category, Purchases.Subcategory, Payments.Amount 
                            FROM Purchases
                            JOIN Payments ON Purchases.PaymentID = Payments.PaymentID
                            WHERE UserID = {userid}
                            ORDER BY Purchases.TimeDate
                            ''', conn)

area_df["TimeDate"] = pd.to_datetime(area_df["TimeDate"])

today = area_df["TimeDate"].max()

timeframes = {

    "1 Month": today - pd.DateOffset(months=1),
    "3 Month": today - pd.DateOffset(months=1),
    "6 Month": today - pd.DateOffset(months=1),
    "1 Year": today - pd.DateOffset(months=1),
    "YTD": pd.Timestamp(today.year, 1, 1),
    "All Time": area_df["TimeDate"].min()
}

categories = sorted(area_df["Category"].unique())

fig = go.Figure()

trace_count = 0
trace_labels = []

for label, start in timeframes.items():

    df_t = area_df[area_df["TimeDate"] >= start]. sort_values("TimeDate")

    if df_t.empty:
        continue

    full_range = pd.date_range()



fig.show()

In [41]:
pie_df = pd.read_sql_query(f'''
                            SELECT Purchases.Category, Purchases.Subcategory, Payments.Amount
                            FROM Purchases
                            JOIN Payments ON Purchases.PaymentID = Payments.PaymentID
                            WHERE UserID = {userid}
                            ORDER BY Purchases.TimeDate
                            ''', conn)

grouped_amount = pie_df.groupby(["Category", "Subcategory"])["Amount"].sum().reset_index(name="total")
grouped_count = pie_df.groupby(["Category", "Subcategory"]).size().reset_index(name="count")

fig = make_subplots(rows = 1, cols = 2,
                     specs=[[{"type": "domain"}, {"type": "domain"}]],
                     subplot_titles=("Total Amount Spent", "Number of Purchases"))

categories = pie_df["Category"].unique()

for i, category in enumerate(categories):
    category_amount_df = grouped_amount[grouped_amount["Category"] == category]
    category_count_df = grouped_count[grouped_count["Category"] == category]
    
    fig.add_trace(

        go.Pie(
            labels = category_amount_df["Subcategory"],
            values = category_amount_df["total"],
            name = f"{category}: Count",
            visible = (i == 0),
            hovertemplate = "%{label}: $%{value}<extra></extra>"
        ), 
        row = 1, col = 1
    )

    fig.add_trace(

        go.Pie(
            labels = category_count_df["Subcategory"],
            values = category_count_df["count"],
            name = f"{category}: Amount",
            visible = (i == 0),
            hovertemplate = "%{label}, %{value}<extra></extra>"
        ),
        row = 1, col = 2
    )

fig.update_layout(

    updatemenus=[
        dict(
            buttons = [
                dict(
                    label = category,
                    method = "update",
                    args = [
                        {"visible": [(j // 2) == i for j in range(2 * len(categories))]},
                        {"title": f"Purchase Breakdown: {category}"}
                    ]
                )
                for i, category in enumerate(categories)
            ],
            direction = "down",
            x = 1,
            y = 1,
        )

    ],
    title = f"Purchase Breakdown: {categories[0]}"
)

fig.show()
    

In [34]:
category_bar_df = pd.read_sql_query(f'''
                  SELECT Purchases.Category, Purchases.Subcategory, Payments.Amount
                  FROM Purchases
                  JOIN Payments ON Purchases.PaymentID = Payments.PaymentID
                  WHERE Purchases.UserID = {userid}
                  ORDER BY Purchases.Category
                  ''', conn)

fig = px.bar(category_bar_df, x = "Category", y = "Amount", color = "Subcategory", title = "Spending per Category")
fig.show()

C:\Users\jduen\AppData\Local\Temp\ipykernel_7384\2047386561.py:1: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



In [23]:
df = pd.read_sql_query(f'''

                        SELECT TOP 5 Purchases.TimeDate, Purchases.Location, Purchases.Category, Purchases.Subcategory, Payments.Amount, Payments.Type
                        FROM Purchases
                        JOIN Payments ON Purchases.PaymentID = Payments.PaymentID
                        JOIN Users ON Purchases.UserID = Users.UserID
                        WHERE Users.UserID = {userid}
                        ORDER BY Purchases.TimeDate DESC
                        ''', conn)

df.head()

C:\Users\jduen\AppData\Local\Temp\ipykernel_13636\1248313436.py:1: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



,TimeDate,Location,Category,Subcategory,Amount,Type
0,2025-10-31 02:55:00,"Columbus, OH",medical,medication,233.34,card
1,2025-09-12 02:55:00,"Charlotte, NC",utilities,water,209.82,cash
2,2025-07-04 02:56:00,"San Diego, CA",health,gym,250.69,card
3,2025-06-27 02:55:00,"Los Angeles, CA",miscellaneous,donations,89.60,cash
4,2025-05-21 02:55:00,"San Antonio, TX",entertainment,events,126.55,cash


In [24]:
fig = go.Figure(data=[go.Table(
    header=dict(values=['Time/Date', 'Location', 'Category', 'SubCategory', 'Amount', 'Type'],
                align='center'),
    cells=dict(values=[df.TimeDate, df.Location, df.Category, df.Subcategory, df.Amount, df.Type],
               align='left')),
])

fig.update_layout(title_text='Your Last 5 Purchases')

fig.show()